In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

In [0]:
CATALOG = 'retail_data'
GOLD_SCHEMA = 'gold'
SILVER_SCHEMA = 'silver'
yest_date = datetime.now() - timedelta(days=1)
yest_date = yest_date.strftime("%Y%m%d")
dbutils.widgets.text("file_run", yest_date, "File Run Parameter")
file_run = dbutils.widgets.get("file_run")

## Customers

In [0]:
spark.sql(
f'''
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.customers
USING DELTA
AS
SELECT
    sha2(mobile_number, 256) AS customer_sk,
    full_name,
    email,
    mobile_number,
    city,
    state,
    zipcode,
    signup_date,
    offline_customer_id,
    online_customer_id,
    file_date,
    source,
    ingestion_ts,
    current_timestamp() AS last_updated_ts
FROM {CATALOG}.{SILVER_SCHEMA}.customers_unified
WHERE 1 = 2;
'''
)

In [0]:
customers_gold = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customers")
customers_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_unified")

In [0]:
spark.sql(
f'''
MERGE INTO {CATALOG}.{GOLD_SCHEMA}.customers t
USING (
    SELECT
        sha2(mobile_number, 256) AS customer_sk,
        full_name,
        email,
        mobile_number,
        city,
        state,
        zipcode,
        signup_date,
        offline_customer_id,
        online_customer_id,
        file_date,
        source,
        ingestion_ts
    FROM {CATALOG}.{SILVER_SCHEMA}.customers_unified
    WHERE file_date = {file_run}
) s
ON t.mobile_number = s.mobile_number

WHEN MATCHED THEN UPDATE SET
    t.customer_sk = s.customer_sk,
    t.full_name = s.full_name,
    t.email = s.email,
    t.city = s.city,
    t.state = s.state,
    t.zipcode = s.zipcode,
    t.signup_date = s.signup_date,
    t.offline_customer_id = s.offline_customer_id,
    t.online_customer_id = s.online_customer_id,
    t.file_date = s.file_date,
    t.source = s.source,
    t.ingestion_ts = s.ingestion_ts,
    t.last_updated_ts = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    customer_sk,
    full_name,
    email,
    mobile_number,
    city,
    state,
    zipcode,
    signup_date,
    offline_customer_id,
    online_customer_id,
    file_date,
    source,
    ingestion_ts,
    last_updated_ts
)
VALUES (
    s.customer_sk,
    s.full_name,
    s.email,
    s.mobile_number,
    s.city,
    s.state,
    s.zipcode,
    s.signup_date,
    s.offline_customer_id,
    s.online_customer_id,
    s.file_date,
    s.source,
    s.ingestion_ts,
    current_timestamp()
);
'''
)

## Products

In [0]:
spark.sql(
f'''
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.products (
    product_sk STRING,
    product_id STRING,
    product_name STRING,
    product_category STRING,
    product_brand STRING,
    product_price FLOAT,
    tax_rate FLOAT,
    currency STRING,
    is_active STRING,
    file_date STRING,
    source STRING,
    ingestion_ts TIMESTAMP,
    effective_from date,
    effective_to date,
    is_current BOOLEAN
)
'''
)

In [0]:
products_gold = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.products")
products_silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.products_unified")

In [0]:
spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW vw_products_current_run AS
select *
,sha2(concat_ws('_', product_id, {file_run}), 256) product_sk
,sha2(concat_ws('_', product_name, product_category, product_brand, product_price, tax_rate, currency, is_active), 256) new_hash
 from {CATALOG}.{SILVER_SCHEMA}.products_unified
 where file_date = {file_run}
"""
)

In [0]:
spark.sql(
f'''
merge into {CATALOG}.{GOLD_SCHEMA}.products t
using vw_products_current_run s
ON t.product_id = s.product_id
AND t.is_current = true

WHEN MATCHED AND
    sha2(concat_ws('_',
        t.product_name,
        t.product_category,
        t.product_brand,
        t.product_price,
        t.tax_rate,
        t.currency,
        t.is_active
    ), 256) <> s.new_hash

THEN UPDATE SET
    t.is_current = false,
    t.effective_to = to_date({file_run} ,'yyyyMMdd')
'''
)

In [0]:
spark.sql(
f'''
insert into {CATALOG}.{GOLD_SCHEMA}.products
SELECT
    s.product_sk,
    s.product_id,
    s.product_name,
    s.product_category,
    s.product_brand,
    s.product_price,
    s.tax_rate,
    s.currency,
    s.is_active,
    s.file_date,
    s.source,
    s.ingestion_ts,
    to_date({file_run} ,'yyyyMMdd') AS effective_from,
    TIMESTAMP '9999-12-31' AS effective_to,
    true AS is_current
FROM vw_products_current_run s
LEFT JOIN {CATALOG}.{GOLD_SCHEMA}.products t
    ON s.product_id = t.product_id
    AND t.is_current = true
WHERE t.product_id IS NULL
   OR sha2(concat_ws('_',
        t.product_name,
        t.product_category,
        t.product_brand,
        t.product_price,
        t.tax_rate,
        t.currency,
        t.is_active
     ), 256) <> s.new_hash
'''
)

## Orders